<a href="https://colab.research.google.com/github/evkoff/DI-Bootcamp-Stage1/blob/main/Wee12/Day5/Tutorial/W12D5_Sentiment_Analysis_with_BERT_using_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 2 : Understanding the BERT Model and Its Role in LLMs and RAG

In this lesson, we will explore the BERT model (Bidirectional Encoder Representations from Transformers), a groundbreaking advancement in natural language processing (NLP). You have already learned about Large Language Models (LLMs) and Retrieval-Augmented Generation (RAG). Now, we will dive deeper into BERT, understand how it works, and see how it connects to LLMs and RAG systems. By the end of this lesson, you will have a clear understanding of BERT's architecture, its applications, and its role in modern NLP systems.

## Pre-training Objective: Masked Language Modeling (MLM)

In [ ]:
! pip install transformers

In [ ]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForMaskedLM.from_pretrained('bert-base-uncased')
text = ("Napoleon revolutionised military organization"
"Napoleon has legacy"
"Legacy still lives"
)
rep = tokenizer(text, return_tensors = "pt")
print("Before Masking",rep.input_ids)
rand = torch.rand(rep.input_ids.shape)
mask_arr = (rand < 0.15) * (rep.input_ids != 101) * (rep.input_ids != 102)
selection = torch.flatten(mask_arr[0].nonzero()).tolist()
rep.input_ids[0, selection] = 103
after_masking = rep.input_ids
print("After Masking",after_masking)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Before Masking tensor([[  101,  8891,  4329,  5084,  2510,  3029,  2532, 15049,  2239,  2038,
          8027, 23115, 15719,  2145,  3268,   102]])
After Masking tensor([[  101,  8891,   103,  5084,  2510,  3029,  2532, 15049,  2239,   103,
          8027, 23115, 15719,  2145,  3268,   102]])


# Hands-on Tutorial : Sentiment Analysis with BERT using Python

---



## Importing Dependencies

We will need the following libraries :

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification

## Downloading The IMDB Dataset

Use the link below to download a condensed subset of the IMDB Movie Reviews dataset—while the full collection includes 50,000 reviews, this version contains just 135.



In [ ]:
df = pd.read_csv(
    'https://gist.githubusercontent.com/Mukilan-Krishnakumar/e998ecf27d11b84fe6225db11c239bc6/raw/74dbac2b992235e555df9a0a4e4d7271680e7e45/imdb_movie_reviews.csv'
)  # Read the IMDB reviews CSV directly from the given URL into a pandas DataFrame named df

# df = df.drop('sentiment', axis=1)  # Remove the existing 'sentiment' column from df, keeping only the review text
df.head()

,text,sentiment
0,"My daughter liked it but I was aghast, that a ...",neg
1,I... No words. No words can describe this. I w...,neg
2,this film is basically a poor take on the old ...,neg
3,"This is a terrible movie, and I'm not even sur...",neg
4,First of all this movie is a piece of reality ...,pos


We will drop the sentiment which comes along with the dataset and predict our own sentiment using BERT

In [ ]:
df_labeled = df.copy()
df = df.drop('sentiment',axis=1)
df.head()

,text
0,"My daughter liked it but I was aghast, that a ..."
1,I... No words. No words can describe this. I w...
2,this film is basically a poor take on the old ...
3,"This is a terrible movie, and I'm not even sur..."
4,First of all this movie is a piece of reality ...


## Model Building and Evaluation

We’ll leverage the bert-base-multilingual-uncased-sentiment model—a version of BERT that’s been fine-tuned specifically for assessing sentiment across multiple languages—to assign emotional valence to each review. To do this, we’ve implemented a helper function called sentiment_movie_score, which iterates over your dataset one row at a time, feeds each review text into the sentiment model, and interprets its output logits as a discrete rating on a 1-to-5 scale (where 1 indicates strongly negative sentiment and 5 indicates strongly positive sentiment). By encapsulating the tokenization, model inference, and score conversion inside this function, you can easily apply consistent, multilingual sentiment scoring to any collection of movie reviews with just a single function call.

In [ ]:
tokenizer = BertTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')
model = BertForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
def sentiment_score(movie_review):
	token = tokenizer.encode(movie_review, return_tensors = 'pt')
	result = model(token)
	return int(torch.argmax(result.logits))+1

In [ ]:
df['sentiment'] = df['text'].apply(lambda x: sentiment_score(x[:512]))

In [ ]:
df.head()

,text,sentiment
0,"My daughter liked it but I was aghast, that a ...",3
1,I... No words. No words can describe this. I w...,1
2,this film is basically a poor take on the old ...,2
3,"This is a terrible movie, and I'm not even sur...",1
4,First of all this movie is a piece of reality ...,4


## Visualising Results

In our experiment tracking setup, we start by initializing a Weights & Biases run with the project name “BERT_Sentiment_Analysis”, which organizes all related metrics and artifacts under a single, easily identifiable workspace. Next, we convert our pandas DataFrame of movie reviews and their predicted sentiment scores into a wandb.Table, allowing us to log the entire dataset as a structured artifact that can be browsed and filtered within the W&B UI. By calling wandb.log({"predictions": table}), we upload this table alongside any scalar metrics or visualizations we’ve collected. Finally, we call run.finish() to mark the end of the logging session—this closes out the run, ensures all data is properly synced, and makes the results available for later inspection and comparison.

In [ ]:
!pip install wandb -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 63.0 MB/s eta 0:00:00


In [ ]:
import wandb
wandb.init(project="BERT_Sentiment_Analysis")
wandb.run.log({"Sentiment Analysis of IMDB Movie Reviews" : wandb.Table(dataframe=df)})
wandb.run.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: raphael-gabbay (raphael-gabbay-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


By wrapping your DataFrame in a wandb.Table, you transform your raw predictions into a first-class artifact that can be browsed, filtered, and charted directly in the Weights & Biases UI. Once logged (e.g. via wandb.log({"predictions": table})), each column becomes a queryable field—so you can slice and dice by sentiment score, review length, or any other feature. The W&B dashboards will automatically generate summary statistics and visualizations (like histograms, scatter plots, or custom charts) based on your table’s schema, making it easy to spot trends or outliers at a glance. For a deeper dive into advanced querying, custom visualizations, and best practices, see the official wandb.Table documentation.



# Fine-Tuning the model further using our small IMDB Dataset

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df_labeled, test_size=0.2, random_state=42)

print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")

Training set size: 108
Validation set size: 27


In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

# Install datasets library if not already installed
try:
    from datasets import Dataset
except ImportError:
    print("Installing 'datasets' library...")
    !pip install datasets -q
    from datasets import Dataset

# 1. Prepare labels: Map 'neg' to 0 and 'pos' to 1
label_map = {'neg': 0, 'pos': 1}
train_df['labels'] = train_df['sentiment'].map(label_map)
val_df['labels'] = val_df['sentiment'].map(label_map)

# Convert pandas DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# 2. Load the pre-trained tokenizer (re-using the one already loaded in the notebook)
# tokenizer = BertTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment') # This is already in kernel state.

# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

# Apply tokenization
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True)

# Select and rename columns for Trainer
# Remove original text column and __index_level_0__ from pandas conversion
tokenized_train_dataset = tokenized_train_dataset.remove_columns(["text", "__index_level_0__"])
tokenized_val_dataset = tokenized_val_dataset.remove_columns(["text", "__index_level_0__"])

# Set format for PyTorch
tokenized_train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 3. Load the pre-trained model for sequence classification with 2 labels
# This will re-initialize the classification head for 2 classes, suitable for 'neg'/'pos' classification.
model_for_finetuning = BertForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment', num_labels=2)

# 4. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,              # total number of training epochs
    per_device_train_batch_size=8,  # batch size per device during training
    per_device_eval_batch_size=8,   # batch size for evaluation
    warmup_steps=50,                 # number of warmup steps for learning rate scheduler (adjusted for small dataset)
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    evaluation_strategy="epoch",     # Evaluate every epoch
    save_strategy="epoch",           # Save checkpoint every epoch
    load_best_model_as_init_checkpoint=True, # Load best model at the end
    metric_for_best_model="eval_loss", # Metric to monitor for early stopping
    greater_is_better=False,         # Lower eval_loss is better
    report_to="none"                 # Disable wandb logging for this Trainer if not explicitly requested
)

# 5. Initialize the Trainer
trainer = Trainer(
    model=model_for_finetuning,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    tokenizer=tokenizer, # Pass tokenizer here for proper data collation
)

# 6. Train the model
print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete!")

# Store the trained model in a global variable
global fine_tuned_model
fine_tuned_model = trainer.model

# Optional: Evaluate the fine-tuned model on the validation set
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

Map:   0%|          | 0/108 [00:00<?, ? examples/s]

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlptown/bert-base-multilingual-uncased-sentiment
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([5]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([5, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


RuntimeError: You set `ignore_mismatched_sizes` to `False`, thus raising an error. For details look at the above report!

Evaluating fine-tuned model on validation set with full metrics...


NameError: name 'trainer' is not defined

## Creating a Gradio Dashboard

In [ ]:
import gradio as gr
import torch
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification

# Ensure the tokenizer and models are loaded (assuming they are from previous cells)
# tokenizer and fine_tuned_model should now be globally available after previous cell execution

# Re-instantiate base model (5-star output)
base_model_5_star = BertForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment', num_labels=5)

# Define prediction function for the original (base) model
def predict_base_model_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
    with torch.no_grad():
        outputs = base_model_5_star(**inputs)
    logits = outputs.logits
    predicted_5_star = torch.argmax(logits, dim=-1).item() + 1 # +1 because original model predicts 0-4
    # Map 5-star prediction to binary: 1,2,3 -> neg (0); 4,5 -> pos (1)
    binary_sentiment = 'Positive' if predicted_5_star >= 4 else 'Negative'
    return f"5-star prediction: {predicted_5_star}, Binary: {binary_sentiment}"

# Define prediction function for the fine-tuned model, accepting the model instance
def predict_finetuned_model_sentiment_with_model(text, model_instance):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
    with torch.no_grad():
        outputs = model_instance(**inputs)
    logits = outputs.logits
    predicted_label = torch.argmax(logits, dim=-1).item() # 0 for neg, 1 for pos
    sentiment = 'Positive' if predicted_label == 1 else 'Negative'
    return sentiment

# Create Gradio interface, passing the fine_tuned_model instance
iface = gr.Interface(
    fn=lambda text: (
        predict_base_model_sentiment(text),
        predict_finetuned_model_sentiment_with_model(text, fine_tuned_model) # Pass the global fine_tuned_model
    ),
    inputs=gr.Textbox(lines=5, label="Enter your movie review"),
    outputs=[
        gr.Textbox(label="Original Model Prediction (5-star & Binary)"),
        gr.Textbox(label="Fine-tuned Model Prediction (Binary)")
    ],
    title="Movie Review Sentiment Analysis Comparison",
    description="Compare sentiment predictions from the original multilingual BERT model and your fine-tuned model."
)

# Launch the interface
iface.launch(debug=True)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e2f1225fbf3fd7ab04.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 393, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2280, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1657, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 65, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^